# nb05 — Embedded Scale: RPi 3/4 and x86 Low-Power
> **~200–300 K parameter models for single-board computers.**  
> Train four featurizer variants, measure RTF, compare embedding quality, and
> simulate streaming inference on a sliding window.

---

## Target hardware

| Platform | RAM | CPU | onnxruntime | RTF target |
|----------|-----|-----|-------------|------------|
| RPi 3B+ | 1 GB | Cortex-A53 × 4 @ 1.4 GHz | OK (armhf wheel) | < 0.05 |
| RPi 4 | 4–8 GB | Cortex-A72 × 4 @ 1.8 GHz | OK | < 0.02 |
| Intel N100 | 8 GB | 4 Gracemont cores @ 3.4 GHz | OK | < 0.01 |
| Intel Celeron | 4 GB | 2–4 cores @ 2.0+ GHz | OK | < 0.03 |

**RTF (Real-Time Factor) = latency / audio_duration.**  
RTF < 0.1 means the model processes audio at least 10× faster than real-time — safe for
continuous always-on detection. Aim for RTF < 0.05 on RPi 3.

## Featurizer comparison

| Tier | Featurizer | What it learns | Strength |
|------|------------|----------------|----------|
| `small` | MFCC-40 | Fixed mel-filterbank + DCT | Fast, deterministic |
| `filterbank_small` | FilterBank | Learnable mel-scale filters | Adapts to language |
| `sincnet_small` | SincNet | Learnable sinc band-pass filters | Good for short keywords |
| `gammatone_small` | Gammatone | Biologically-inspired auditory filters | Robust to noise |

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD         = os.environ.get("WAKE_WORD",         "hey jarvis")
OUTPUT_DIR        = os.environ.get("OUTPUT_DIR",        "./ww_output")
DEVICE            = os.environ.get("DEVICE",            "auto")
SEED              = int(os.environ.get("SEED",          "42"))

# ── Tiers to train ────────────────────────────────────────────────────────────
TIERS_TO_TRAIN    = os.environ.get(
    "TIERS_TO_TRAIN", "small,filterbank_small,sincnet_small,gammatone_small"
).split(",")
EPOCHS            = int(os.environ.get("EPOCHS",        "30"))
BATCH_SIZE        = int(os.environ.get("BATCH_SIZE",    "16"))
SKIP_COMPLETED    = os.environ.get("SKIP_COMPLETED",    "true").lower() == "true"

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE        = int(os.environ.get("N_POSITIVE",    "500"))
LANG              = os.environ.get("LANG",              "en")
ADVERSARIAL       = os.environ.get("ADVERSARIAL",       "true").lower() == "true"
DOWNLOAD_AUGMENT  = os.environ.get("DOWNLOAD_AUGMENT",  "true").lower() == "true"
REUSE_DATASET     = os.environ.get("REUSE_DATASET",     "true").lower() == "true"
CUSTOM_TRAIN_CSV  = os.environ.get("CUSTOM_TRAIN_CSV",  "")
CUSTOM_TEST_CSV   = os.environ.get("CUSTOM_TEST_CSV",   "")

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI        = os.environ.get("MLFLOW_URI",        "")
MLFLOW_SECRET     = os.environ.get("MLFLOW_SECRET",     "MLFLOW_TOKEN")

print(f"Tiers    : {TIERS_TO_TRAIN}")
print(f"Epochs   : {EPOCHS}  |  Batch: {BATCH_SIZE}")
print(f"Wake word: {WAKE_WORD!r}")

In [ ]:
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)

import torch
torch.set_num_threads(min(12, os.cpu_count() or 4))
os.environ.setdefault("OMP_NUM_THREADS", str(min(12, os.cpu_count() or 4)))

if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print(f"MLflow token injected from Kaggle Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"No Kaggle secret found: {e}")
if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI

print(f"Platform : {_platform} | CUDA: {torch.cuda.is_available()} | threads: {torch.get_num_threads()}")

In [ ]:
import shutil
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 4, f"Only {free_gb:.1f} GB free — need at least 4 GB."
print(f"Disk free: {free_gb:.1f} GB")

_aug_kwargs_full = {}

if CUSTOM_TRAIN_CSV:
    import random, csv as _csv
    from ww_trainer.utils import read_dataset_csv
    train_csv = Path(CUSTOM_TRAIN_CSV)
    test_csv  = Path(CUSTOM_TEST_CSV) if CUSTOM_TEST_CSV else None
    if test_csv is None:
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED); random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            for p, rs in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(p, "w", newline="") as f:
                    _csv.writer(f).writerows(rs)
        train_csv, test_csv = split_train, split_test
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word
    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"
    if REUSE_DATASET and _train_csv_check.exists():
        print(f"Reusing dataset at {dataset_dir}")
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
    else:
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD,
            output_dir=dataset_dir,
            n_positive=N_POSITIVE,
            lang=LANG,
            adversarial=ADVERSARIAL,
            vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT,
            seed=SEED,
        ))
    train_csv, test_csv = _dr.train_csv, _dr.test_csv
    for attr, key in [("bg_noise_dir", "bg_noise_folder"),
                      ("music_dir", "music_folder"),
                      ("rir_dir", "rir_folder")]:
        d = getattr(_dr, attr, None)
        if d and Path(d).exists():
            _aug_kwargs_full[key] = str(d)

print(f"train_csv: {train_csv}")
print(f"test_csv : {test_csv}")

In [ ]:
import json, time
from pathlib import Path
from ww_trainer.quickstart import train_from_wakeword

results_dir = Path(OUTPUT_DIR) / "embedded_results"
results_dir.mkdir(parents=True, exist_ok=True)
all_results = []

for tier in TIERS_TO_TRAIN:
    result_file = results_dir / f"{tier}.json"
    model_subdir = Path(OUTPUT_DIR) / "models" / tier

    print(f"\n{'='*60}\nTier: {tier!r}")
    if SKIP_COMPLETED and result_file.exists():
        saved = json.loads(result_file.read_text())
        print(f"  SKIP: F1={saved.get('f1', 0):.4f}")
        all_results.append(saved)
        continue

    t0 = time.time()
    try:
        r = train_from_wakeword(
            WAKE_WORD, str(model_subdir),
            tier=tier, epochs=EPOCHS, batch_size=BATCH_SIZE,
            device=DEVICE, seed=SEED, reuse_dataset=True,
            **_aug_kwargs_full,
        )
        elapsed = time.time() - t0
        row = {
            "tier": tier,
            "f1": r.metrics.get("f1", 0.0),
            "precision": r.metrics.get("precision", 0.0),
            "recall": r.metrics.get("recall", 0.0),
            "elapsed_s": round(elapsed, 1),
            "head_onnx": str(r.best_onnx_path) if r.best_onnx_path else "",
            "feat_onnx": str(model_subdir / "model" / "best_f1_featurizer.onnx"),
            "status": "ok",
        }
        print(f"  DONE: F1={row['f1']:.4f}  ({elapsed:.0f}s)")
    except Exception as exc:
        elapsed = time.time() - t0
        row = {
            "tier": tier, "f1": 0.0, "precision": 0.0, "recall": 0.0,
            "elapsed_s": round(elapsed, 1), "head_onnx": "", "feat_onnx": "",
            "status": f"error: {exc}",
        }
        print(f"  ERROR: {exc}")

    result_file.write_text(json.dumps(row, indent=2))
    all_results.append(row)

print(f"\nDone: {sum(1 for r in all_results if r['status']=='ok')}/{len(all_results)} succeeded.")

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

# ── Latency and RTF benchmark ─────────────────────────────────────────────────
# RTF = latency_per_1s_audio / 1.0
# RTF < 0.1 is the minimum for real-time embedded use.
# Aim for RTF < 0.05 on RPi 3.

try:
    from ww_trainer.benchmark import measure_latency
    _have_bench = True
except ImportError:
    _have_bench = False

bench_rows = []
for row in all_results:
    if row["status"] != "ok":
        bench_rows.append({"tier": row["tier"], "latency_ms": None, "rtf": None, "f1": row["f1"]})
        continue
    tier = row["tier"]
    feat_path = Path(row["feat_onnx"])
    head_path = Path(row["head_onnx"])
    if not feat_path.exists() or not head_path.exists():
        bench_rows.append({"tier": tier, "latency_ms": None, "rtf": None, "f1": row["f1"]})
        continue

    if _have_bench:
        try:
            lat_ms = measure_latency(str(feat_path), str(head_path), n_runs=20)
        except Exception as e:
            lat_ms = None
            print(f"  {tier}: bench error: {e}")
    else:
        from ww_trainer.inference import OnnxWakeWordInferencer
        inf = OnnxWakeWordInferencer(str(feat_path), str(head_path))
        dummy = np.random.randn(16000).astype(np.float32)
        for _ in range(3): inf.infer(dummy)  # warm-up
        t0 = time.perf_counter()
        for _ in range(20): inf.infer(dummy)
        lat_ms = (time.perf_counter() - t0) / 20 * 1000

    rtf = (lat_ms / 1000) if lat_ms is not None else None
    ok_marker = "OK" if (rtf is not None and rtf < 0.1) else "SLOW"
    print(f"  {tier:25s}: {lat_ms:.1f} ms  RTF={rtf:.5f}  [{ok_marker}]" if lat_ms else f"  {tier}: n/a")
    bench_rows.append({"tier": tier, "latency_ms": round(lat_ms, 2) if lat_ms else None,
                       "rtf": round(rtf, 5) if rtf else None, "f1": row["f1"]})

df_bench = pd.DataFrame(bench_rows)
print()
print(df_bench.to_string(index=False))
print("\nRTF < 0.1 → safe for real-time embedded | RTF < 0.05 → safe for RPi 3")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv
import torchaudio
import torch
from pathlib import Path
from sklearn.decomposition import PCA

# ── PCA of embeddings for each featurizer ─────────────────────────────────────
# Load up to 200 test samples, run through each featurizer's ONNX,
# project to 2D with PCA, and colour by label.
# A well-separated PCA plot means the featurizer has useful discriminative info.

import onnxruntime as ort

def _load_test_samples(csv_path, max_samples=200):
    """Returns (list of wav_np, list of labels)."""
    wavs, labels = [], []
    with open(csv_path) as f:
        for row in csv.reader(f):
            if len(row) < 2 or not Path(row[0]).exists():
                continue
            wav, sr = torchaudio.load(row[0])
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            wavs.append(wav.mean(0).numpy().astype(np.float32))
            labels.append(int(row[1].strip()))
            if len(wavs) >= max_samples:
                break
    return wavs, labels

def _extract_embeddings(feat_onnx_path, wavs):
    """Run each wav through the featurizer ONNX and return (N, D) array."""
    sess = ort.InferenceSession(str(feat_onnx_path),
                                providers=["CPUExecutionProvider"])
    in_name = sess.get_inputs()[0].name
    embs = []
    for wav in wavs:
        inp = wav[np.newaxis, :]  # (1, T)
        out = sess.run(None, {in_name: inp})[0]  # (1, T', D) or (1, D)
        embs.append(out.reshape(out.shape[-1]) if out.ndim == 2 else out.mean(axis=1).ravel())
    return np.array(embs)

print("Loading test samples...")
wavs, labels = _load_test_samples(test_csv, max_samples=200)
labels = np.array(labels)
print(f"  {len(wavs)} samples: {(labels==1).sum()} positive, {(labels==0).sum()} negative")

ok_rows = [r for r in all_results if r["status"] == "ok" and Path(r["feat_onnx"]).exists()]
n_plots = len(ok_rows)

if n_plots == 0:
    print("No successful tiers with featurizer ONNX — skipping PCA.")
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    fig.suptitle(f"PCA of featurizer embeddings — {WAKE_WORD!r}", fontsize=12)

    for ax, row in zip(axes, ok_rows):
        tier = row["tier"]
        print(f"  Extracting embeddings: {tier}...")
        try:
            embs = _extract_embeddings(row["feat_onnx"], wavs)
            pca = PCA(n_components=2, random_state=SEED)
            coords = pca.fit_transform(embs)
            for lbl, color, marker, name in [
                (1, "steelblue", "o", "wake"),
                (0, "coral", "x", "non-wake")
            ]:
                mask = labels == lbl
                ax.scatter(coords[mask, 0], coords[mask, 1],
                           c=color, marker=marker, s=20, alpha=0.6, label=name)
            ax.set_title(f"{tier}\nvar: {pca.explained_variance_ratio_.sum():.2f}")
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
            ax.legend(fontsize=8)
        except Exception as e:
            ax.set_title(f"{tier}\n(error)")
            print(f"  ERROR for {tier}: {e}")

    plt.tight_layout()
    pca_path = str(Path(OUTPUT_DIR) / "embedded_pca.png")
    plt.savefig(pca_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"PCA plot saved: {pca_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Results comparison table + bar chart ──────────────────────────────────────

df_res  = pd.DataFrame(all_results)
df_bench_merged = df_bench.copy()
df = df_res.merge(df_bench_merged[["tier","latency_ms","rtf"]], on="tier", how="left")

# Add ONNX sizes
def _kb(path_str):
    p = Path(path_str)
    return round(p.stat().st_size / 1024, 1) if p.exists() else None

df["head_kb"] = df["head_onnx"].apply(_kb)
df["feat_kb"] = df["feat_onnx"].apply(_kb)

_cols = ["tier", "f1", "precision", "recall", "latency_ms", "rtf", "head_kb", "feat_kb", "status"]
_cols = [c for c in _cols if c in df.columns]
print("Embedded tier comparison:")
print(df[_cols].to_string(index=False))

df_ok = df[df["status"] == "ok"].copy()
if not df_ok.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"Embedded tiers — {WAKE_WORD!r}", fontsize=12)

    axes[0].bar(df_ok["tier"], df_ok["f1"], color="steelblue", edgecolor="white")
    axes[0].set_ylabel("F1")
    axes[0].set_title("F1 by featurizer")
    axes[0].set_ylim(0, 1.05)
    axes[0].tick_params(axis="x", rotation=30)

    rtf_vals = pd.to_numeric(df_ok["rtf"], errors="coerce")
    axes[1].bar(df_ok["tier"], rtf_vals, color="coral", edgecolor="white")
    axes[1].axhline(0.1, color="red", linestyle="--", linewidth=1)
    axes[1].text(len(df_ok) - 0.5, 0.105, "RTF limit (0.1)", fontsize=8, color="red")
    axes[1].set_ylabel("RTF (lower = faster)")
    axes[1].set_title("Real-time factor")
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plot_path = str(Path(OUTPUT_DIR) / "embedded_results.png")
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {plot_path}")

df.to_csv(Path(OUTPUT_DIR) / "embedded_results.csv", index=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torchaudio
import csv
from pathlib import Path

# ── Streaming inference simulation ────────────────────────────────────────────
# Chunk a full test audio into 512-sample (32 ms) frames.
# Accumulate frames until we have 1 second of audio, then run inference.
# Plot the confidence score over time — should spike when the wake word is spoken.
#
# This shows what always-on detection looks like in practice.

# Find best tier
best_row = sorted(
    [r for r in all_results if r["status"] == "ok"],
    key=lambda r: r.get("f1", 0), reverse=True
)
best_row = best_row[0] if best_row else None

if best_row is None or not Path(best_row.get("feat_onnx", "")).exists():
    print("No successful tier with ONNX files — skipping streaming demo.")
else:
    # Find a positive sample (ideally one with context before/after)
    _pos_path = None
    with open(test_csv) as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
                _pos_path = row[0]
                break

    if _pos_path is None:
        print("No positive sample for streaming demo.")
    else:
        from ww_trainer.inference import OnnxWakeWordInferencer
        inferencer = OnnxWakeWordInferencer(
            best_row["feat_onnx"], best_row["head_onnx"]
        )

        wav, sr = torchaudio.load(_pos_path)
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)
        wav_np = wav.mean(0).numpy().astype(np.float32)

        FRAME_SIZE  = 512   # samples (32 ms at 16 kHz)
        WINDOW_SIZE = 16000 # 1 second sliding window

        scores = []
        times  = []
        buffer = np.zeros(WINDOW_SIZE, dtype=np.float32)

        for i in range(0, len(wav_np) - FRAME_SIZE + 1, FRAME_SIZE):
            frame = wav_np[i : i + FRAME_SIZE]
            buffer = np.roll(buffer, -FRAME_SIZE)
            buffer[-FRAME_SIZE:] = frame
            score = inferencer.infer(buffer)
            scores.append(score)
            times.append((i + FRAME_SIZE) / 16000)

        fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
        t_axis = np.linspace(0, len(wav_np) / 16000, len(wav_np))
        axes[0].plot(t_axis, wav_np, linewidth=0.5, color="gray")
        axes[0].set_ylabel("Amplitude")
        axes[0].set_title(f"Streaming inference — {best_row['tier']!r} on {Path(_pos_path).name}")
        axes[1].plot(times, scores, color="steelblue", linewidth=1.5)
        axes[1].axhline(0.5, color="red", linestyle="--", linewidth=1, label="threshold=0.5")
        axes[1].fill_between(times, 0, scores, alpha=0.2, color="steelblue")
        axes[1].set_ylabel("Wake-word confidence")
        axes[1].set_xlabel("Time (s)")
        axes[1].set_ylim(-0.05, 1.05)
        axes[1].legend(fontsize=8)

        plt.tight_layout()
        stream_path = str(Path(OUTPUT_DIR) / "streaming_confidence.png")
        plt.savefig(stream_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Streaming plot saved: {stream_path}")
        peak = max(scores) if scores else 0
        print(f"Peak score: {peak:.4f}  at t={times[scores.index(peak)]:.2f}s" if scores else "")

In [ ]:
import csv
import numpy as np
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

# ── ONNX verification + inference test ───────────────────────────────────────

print("ONNX file check:")
for row in all_results:
    tier = row["tier"]
    feat = Path(row["feat_onnx"]) if row["feat_onnx"] else None
    head = Path(row["head_onnx"]) if row["head_onnx"] else None
    feat_ok = feat and feat.exists()
    head_ok = head and head.exists()
    feat_kb = f"{feat.stat().st_size/1024:.0f} KB" if feat_ok else "MISSING"
    head_kb = f"{head.stat().st_size/1024:.0f} KB" if head_ok else "MISSING"
    status_icon = "OK" if (feat_ok and head_ok) else "INCOMPLETE"
    print(f"  [{status_icon}] {tier:25s}  feat={feat_kb}  head={head_kb}")

# Run inference with best tier
best_row = sorted(
    [r for r in all_results if r["status"] == "ok"],
    key=lambda r: r.get("f1", 0), reverse=True
)
best_row = best_row[0] if best_row else None

print()
if best_row and Path(best_row.get("feat_onnx", "")).exists():
    inferencer = OnnxWakeWordInferencer(best_row["feat_onnx"], best_row["head_onnx"])
    _pos_path = None
    with open(test_csv) as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
                _pos_path = row[0]
                break
    if _pos_path:
        wav, sr = torchaudio.load(_pos_path)
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)
        score = inferencer.infer(wav.mean(0).numpy().astype(np.float32))
        print(f"Inference test ({best_row['tier']!r}): score={score:.4f}  "
              f"({'PASS' if score > 0.5 else 'LOW'})")

print()
print("=" * 60)
print("CLI commands:")
if best_row:
    print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
    print(f"      --featurizer {best_row['feat_onnx']} \\")
    print(f"      --model      {best_row['head_onnx']} \\")
    print(f"      --audio      sample.wav")
    models_dir = Path(OUTPUT_DIR) / "models"
    print(f"  .venv/bin/python scripts/eval/listen_all.py --models-dir {models_dir} --max-models 5")
print("=" * 60)